# Geometric Transforms: Image Scaling and Interpolation

This notebook documents low-level implementations of spatial resampling—algorithms that change image resolution.

The project compares scaling methods analytically and by performance:
- **Naive scaling (no interpolation)** — direct coordinate mapping.
- **Nearest Neighbor Interpolation** — geometric resampling without blending.
- **Bilinear Interpolation** — sub-pixel values from four neighboring samples.
- **OpenCV interpolation benchmark** (including Bicubic and Lanczos4).
- **Tonal resolution analysis (quantization)** — effect of bit-depth reduction on perceived image quality.

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

# Load required test data
if not os.path.exists("parrot.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/parrot.bmp --no-check-certificate
if not os.path.exists("clock.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/clock.bmp --no-check-certificate
if not os.path.exists("chessboard.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/chessboard.bmp --no-check-certificate
if not os.path.exists("lena.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/lena.bmp --no-check-certificate
if not os.path.exists("firetruck.jpg") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/firetruck.jpg --no-check-certificate

I_chessboard = cv2.imread('chessboard.bmp')            
I_chessboard_RGB = cv2.cvtColor(I_chessboard, cv2.COLOR_BGR2GRAY) 

plt.figure(figsize=(I_chessboard_RGB.shape[0]/100,I_chessboard_RGB.shape[1]/100), dpi=200)
plt.imshow(I_chessboard_RGB, cmap ="gray")
plt.xticks([]), plt.yticks([]) 
plt.show()

## 1. Geometric Scaling (No Interpolation)

Target coordinates are mapped back to the source by multiplying by scale factors and truncating to integers. Without averaging, this produces strong aliasing artifacts.

In [ ]:
def scale(I, x_scale, y_scale):
    height, width = I.shape
    new_height = int(height * y_scale)
    new_width = int(width * x_scale)

    scaled_image = np.zeros((new_height, new_width), dtype=I.dtype)
    
    for y in range(height):
        for x in range(width):
            new_y = int(y * y_scale)
            new_x = int(x * x_scale)
            if new_y < new_height and new_x < new_width:
                scaled_image[new_y, new_x] = I[y, x]

    return scaled_image


In [ ]:
scaled_i_1_1 = scale(I_chessboard_RGB, 1, 1)
plt.figure(figsize=(scaled_i_1_1.shape[0]/100,scaled_i_1_1.shape[1]/100), dpi=200)
plt.imshow(scaled_i_1_1, cmap ="gray")
plt.xticks([]), plt.yticks([])
plt.show()
scale(I_chessboard_RGB, 1, 1)

In [ ]:
scaled_i_3_2 = scale(I_chessboard_RGB, 3.0, 2.0)
plt.figure(figsize=(scaled_i_3_2.shape[0]/100,scaled_i_3_2.shape[1]/100), dpi=200)
plt.imshow(scaled_i_3_2, cmap ="gray")
plt.xticks([]), plt.yticks([])
plt.show()

scale(I_chessboard_RGB, 3, 2)


In [ ]:
scaled_i_0_5_0_5 = scale(I_chessboard_RGB, 0.5, 0.5)
plt.figure(figsize=(scaled_i_0_5_0_5.shape[0]/100,scaled_i_0_5_0_5.shape[1]/100), dpi=200)
plt.imshow(scaled_i_0_5_0_5, cmap ="gray")
plt.xticks([]), plt.yticks([])
plt.show()
scale(I_chessboard_RGB, 0.5, 0.5)

## 2. Nearest Neighbor Interpolation

This method avoids dropping pixels during transforms. Instead of mapping each source pixel forward, the algorithm iterates over the output grid and samples the nearest source coordinate for every destination pixel.

In [ ]:
I = cv2.imread('parrot.bmp')
I = cv2.cvtColor(I, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(I.shape[0]/100,I.shape[1]/100), dpi=200)
plt.imshow(I, cmap ="gray")
plt.xticks([]), plt.yticks([])
plt.show()

In [ ]:
def interpolate_nearest_neighbour(I, x_scale, y_scale):
    height, width = I.shape

    new_height = int(height * y_scale)
    new_width = int(width * x_scale)

    scaled_image = np.zeros((new_height, new_width), dtype=I.dtype)
    
    for y in range(new_height):
        for x in range(new_width):
            orig_y = min(int(y / y_scale), height - 1)
            orig_x = min(int(x / x_scale), width - 1)
            scaled_image[y, x] = I[orig_y, orig_x]

    return scaled_image


In [ ]:
I_chess = cv2.cvtColor(cv2.imread('chessboard.bmp'), cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(10,3))
plt.subplot(131)
plt.imshow(scale(I_chess, 1, 1), cmap="gray")
plt.subplot(132)
plt.imshow(scale(I_chess, 3, 2), cmap="gray")
plt.subplot(133)
plt.imshow(scale(I_chess, 0.5, 0.5), cmap="gray")
plt.show()

In [ ]:
I_parrot = cv2.imread('parrot.bmp', cv2.IMREAD_GRAYSCALE)
I_clock = cv2.imread('clock.bmp', cv2.IMREAD_GRAYSCALE)
I_chess = cv2.imread('chessboard.bmp', cv2.IMREAD_GRAYSCALE)

fig, ax = plt.subplots(2, 3, figsize=(10, 6))
ax[0,0].imshow(interpolate_nearest_neighbour(I_parrot, 1.5, 1.5), cmap='gray')
ax[0,1].imshow(interpolate_nearest_neighbour(I_parrot, 2.5, 2.5), cmap='gray')
ax[0,2].imshow(interpolate_nearest_neighbour(I_parrot, 1.5, 2.5), cmap='gray')
ax[1,0].imshow(interpolate_nearest_neighbour(I_parrot, 0.5, 0.5), cmap='gray')
ax[1,1].imshow(interpolate_nearest_neighbour(I_clock, 1.5, 1.5), cmap='gray')
ax[1,2].imshow(interpolate_nearest_neighbour(I_chess, 1.5, 1.5), cmap='gray')
plt.show()

## 3. Bilinear Interpolation

To reduce blocky edges, bilinear interpolation estimates sub-pixel intensity as a weighted blend of four surrounding source samples, with weights proportional to distance.

The implementation handles edge cases and out-of-bounds access for border pixels using `floor()` and `ceil()`.

In [ ]:
def double_linear_interpolation(I, x_scale, y_scale):
    height, width = I.shape
    new_height = int(height * y_scale)
    new_width = int(width * x_scale)
    
    scaled = np.zeros((new_height, new_width), dtype=np.uint8)
    I_f = I.astype(np.float32) 
    
    for y in range(new_height):
        for x in range(new_width):
            orig_y = y / y_scale
            orig_x = x / x_scale
            
            y0 = int(np.floor(orig_y))
            y1 = min(int(np.ceil(orig_y)), height - 1)
            x0 = int(np.floor(orig_x))
            x1 = min(int(np.ceil(orig_x)), width - 1)
            
            if y0 == y1 and x0 == x1:
                scaled[y, x] = np.uint8(I_f[y0, x0])
            elif y0 == y1:
                scaled[y, x] = np.uint8((I_f[y0, x0] * (x1 - orig_x) + I_f[y0, x1] * (orig_x - x0)) / (x1 - x0))
            elif x0 == x1:
                scaled[y, x] = np.uint8((I_f[y0, x0] * (y1 - orig_y) + I_f[y1, x0] * (orig_y - y0)) / (y1 - y0))
            else:
                top = (I_f[y0, x0] * (x1 - orig_x) + I_f[y0, x1] * (orig_x - x0)) / (x1 - x0)
                bot = (I_f[y1, x0] * (x1 - orig_x) + I_f[y1, x1] * (orig_x - x0)) / (x1 - x0)
                scaled[y, x] = np.uint8((top * (y1 - orig_y) + bot * (orig_y - y0)) / (y1 - y0))

    return scaled

fig, ax = plt.subplots(2, 3, figsize=(10, 6))
ax[0,0].imshow(double_linear_interpolation(I_parrot, 1.5, 1.5), cmap='gray')
ax[0,1].imshow(double_linear_interpolation(I_parrot, 2.5, 2.5), cmap='gray')
ax[0,2].imshow(double_linear_interpolation(I_parrot, 1.5, 2.5), cmap='gray')
ax[1,0].imshow(double_linear_interpolation(I_parrot, 0.5, 0.5), cmap='gray')
ax[1,1].imshow(double_linear_interpolation(I_clock, 1.5, 1.5), cmap='gray')
ax[1,2].imshow(double_linear_interpolation(I_chess, 1.5, 1.5), cmap='gray')
plt.show()

## 4. OpenCV Scaling Benchmark

This experiment benchmarks OpenCV's built-in interpolation methods for runtime and visual quality. It highlights the trade-off between sharpness when upscaling (Lanczos4) and computational cost (fastest with Nearest).

In [ ]:
from timeit import default_timer as timer

I_firetruck = cv2.imread('firetruck.jpg', cv2.IMREAD_GRAYSCALE)

methods = [
    ("NEAREST", cv2.INTER_NEAREST),
    ("LINEAR", cv2.INTER_LINEAR),
    ("CUBIC", cv2.INTER_CUBIC),
    ("AREA", cv2.INTER_AREA),
    ("LANCZOS4", cv2.INTER_LANCZOS4)
]

fx, fy = 3.5, 4.5
fig, axs = plt.subplots(1, 5, figsize=(25, 5))

for idx, (name, method) in enumerate(methods):
    start = timer()
    res = cv2.resize(I_firetruck, (0, 0), fx=fx, fy=fy, interpolation=method)
    end = timer()
    
    axs[idx].imshow(res, cmap='gray')
    axs[idx].set_title(f"{name}\nTime: {end - start:.5f} s")
    axs[idx].axis('off')

plt.show()


## 5. Interpolation and Physical Image Size

This experiment illustrates display/print resolution (DPI — dots per inch). The input image is downsampled to progressively smaller arrays, then rendered at a fixed physical extent (forced stretch via `extent`). The loss of detail is clearly visible.

In [ ]:
I_lena = cv2.imread('lena.bmp', cv2.IMREAD_GRAYSCALE)

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(I_lena, cmap='gray', extent=[0, 512, 512, 0])
ax[1].imshow(cv2.resize(I_lena, (256, 256)), cmap='gray', extent=[0, 512, 512, 0])
ax[2].imshow(cv2.resize(I_lena, (128, 128)), cmap='gray', extent=[0, 512, 512, 0])
ax[3].imshow(cv2.resize(I_lena, (64, 64)), cmap='gray', extent=[0, 512, 512, 0])
plt.show()


## 6. Intensity Level Quantization (Tonal Resolution)

Standard images store 256 gray levels (8-bit). This experiment probes perceptual limits by aggressively reducing tonal depth—from 5-bit (32 levels) down to 1-bit binarization—producing visible posterization.

In [ ]:
I_lena = cv2.imread('lena.bmp', cv2.IMREAD_GRAYSCALE)

I_31 = np.zeros_like(I_lena)
cv2.normalize(I_lena, I_31, 0, 31, cv2.NORM_MINMAX)

I_15 = np.zeros_like(I_lena)
cv2.normalize(I_lena, I_15, 0, 15, cv2.NORM_MINMAX)

I_7 = np.zeros_like(I_lena)
cv2.normalize(I_lena, I_7, 0, 7, cv2.NORM_MINMAX)

I_3 = np.zeros_like(I_lena)
cv2.normalize(I_lena, I_3, 0, 3, cv2.NORM_MINMAX)

I_1 = np.zeros_like(I_lena)
cv2.normalize(I_lena, I_1, 0, 1, cv2.NORM_MINMAX)

fig, ax = plt.subplots(2, 3, figsize=(10, 6))
ax[0,0].imshow(I_lena, cmap='gray', vmin=0, vmax=255)
ax[0,0].set_title("Original (8-bit)")
ax[0,1].imshow(I_31, cmap='gray', vmin=0, vmax=31)
ax[0,1].set_title("5-bit (32 levels)")
ax[0,2].imshow(I_15, cmap='gray', vmin=0, vmax=15)
ax[0,2].set_title("4-bit (16 levels)")
ax[1,0].imshow(I_7, cmap='gray', vmin=0, vmax=132)
ax[1,0].set_title("3-bit (8 levels)")
ax[1,1].imshow(I_3, cmap='gray', vmin=0, vmax=64)
ax[1,1].set_title("2-bit (4 levels)")
ax[1,2].imshow(I_1, cmap='gray', vmin=0, vmax=1)
ax[1,2].set_title("1-bit (Binarization)")
plt.show()


### Cleaning Temporary Files

Prevents downloaded test images (BMP, JPG) from persisting in the project workspace.

In [ ]:
import os
import glob

trash_files = glob.glob('*.bmp*') + glob.glob('*.jpg*') + glob.glob('*.png*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Cleanup complete! Workspace is clear.")